# 16 · Neural-network challenger (Keras MLP vs. LightGBM)

This notebook trains **one feed-forward neural network (MLP)** on the same 39 features as our final LightGBM and compares the two on the same clients with the same metric. The rules come from the **frozen** [model-comparison-plan.md](model-comparison-plan.md) (frozen 2026-11-05) and the decisions in [protocol.md](protocol.md).

**Plan changes** (both logged in `protocol.md` on 2026-11-05, and to be listed in the report):
1. **Feature tables.** We use the v3 tables as rebuilt on Dhanya's machine (Apple M2). Their SHA-256 values differ from the plan, but the labels (`y_train`, `y_valid`) and therefore the clients and the split are identical. The likely cause is tiny floating-point differences between Intel and Apple-silicon CPUs. Agreed with Tim.
2. **Hardware.** Training and timing run on an Apple M2 (8 cores, 8 GB RAM, CPU only) instead of the Intel i7 named in the plan. Sections 1–4 of this notebook were written by Dhanya's Claude session, because Tim's draft was never pushed.

**The most important rule:** the 27,099 test clients (`X_valid_v3.csv` / `y_valid.csv`) are used **exactly once**, in section 7, after the final configuration is saved and committed. They are never used for early stopping, tuning, architecture choice, thresholds or preprocessing.

| Section | Content |
|---|---|
| 1 | Setup: imports, seeds, paths, versions |
| 2 | Data check: hashes, row counts, fraud rate, feature list, test ids |
| 3 | Preprocessing (fitted on training clients only) |
| 4 | Model design and a short smoke test |
| 5 | Random search (30 trials, inner 80/20 split of the training clients) |
| 6 | Final configuration (saved and committed before section 7) |
| 7 | Test evaluation, scored once (seeds 42, 1, 2) |
| 8 | Timing: NN and LightGBM (LightGBM timing only, no scoring) |
| 9 | Record of the environment |

Run this notebook with the **NN environment** `.venv-nn` (Python 3.12, TensorFlow 2.16.2), from the repository root.

## 1 · Setup

We fix all random seeds, define the paths and print the library versions. Seed 42 is used for the search; the final model is trained with seeds 42, 1 and 2 (same seeds as LightGBM).

We also switch on TensorFlow's **op determinism**, so that the same seed gives the same network on the same machine. On a CPU this costs very little speed.

In [1]:
import hashlib
import json
import os
import platform
import time
from pathlib import Path

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # hide TensorFlow's info messages, keep warnings and errors

import numpy as np
import pandas as pd
import sklearn
import tensorflow as tf
import keras
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight

# ---- paths (the notebook lives in the repository root) ----
FEATURE_DIR = Path("data/features")
MODEL_DIR = Path("models")            # NN model files go to models/nn_* (ignored by git)
RESULTS_DIR = Path("results")         # small result files, committed
RESULTS_DIR.mkdir(exist_ok=True)

# ---- seeds ----
SEARCH_SEED = 42            # inner split, trial sampling and all search fits
FINAL_SEEDS = [42, 1, 2]    # final refits in section 7 (same seeds as LightGBM)

# Same seed -> same weights, same dropout masks, same batch order (on this machine)
tf.config.experimental.enable_op_determinism()
keras.utils.set_random_seed(SEARCH_SEED)

NOTEBOOK_START = time.perf_counter()  # for the total runtime in section 9

print("Python      ", platform.python_version())
print("TensorFlow  ", tf.__version__)
print("Keras       ", keras.__version__)
print("scikit-learn", sklearn.__version__)
print("pandas      ", pd.__version__)
print("numpy       ", np.__version__)
print("Devices     ", [d.device_type for d in tf.config.list_physical_devices()])

Python       3.12.14
TensorFlow   2.16.2
Keras        3.15.1
scikit-learn 1.9.1
pandas       3.0.6
numpy        1.26.4
Devices      ['CPU']


## 2 · Data check

We load the **training** tables (`X_train_v3`, `y_train`) and check them against the numbers in the plan:

* 108,394 training clients, 6,053 fraud (5.584%);
* the 39 features of `models/selected_features_v3.json`, **in that order**, the same as LightGBM;
* `client_id` is the index and `target` is not a feature.

Of the **test** tables we only read the **client ids** of `X_valid_v3.csv` here, to check that they are exactly the clients of `y_valid.csv` (the clients LightGBM was scored on). The test features are loaded for the first time in section 7. The test labels are only counted here (27,099 clients, 1,513 fraud), which is a fixed fact from the plan and is not used for any modelling choice.

The SHA-256 values are compared with the hashes of the rebuilt tables logged in `protocol.md` (plan change 1).

In [2]:
def sha16(path):
    # first 16 hex characters of the SHA-256 of a file, as in the plan
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()[:16]

# Expected hashes: rebuilt tables (protocol.md, 2026-11-05, plan change). Labels = same as in the frozen plan.
EXPECTED_SHA = {
    "X_train_v3.csv": "1cf6b683d2da1516",
    "X_valid_v3.csv": "5f880604ffea81e0",
    "y_train.csv":    "5174ffcfe33498d9",
    "y_valid.csv":    "1d12f28fd072ac98",
}
# Hashes written in the frozen plan (Tim's Intel machine), shown for reference only
PLAN_SHA = {"X_train_v3.csv": "90e52789a4c4698a", "X_valid_v3.csv": "1a5ebdca6d5f2c2f",
            "y_train.csv": "5174ffcfe33498d9", "y_valid.csv": "1d12f28fd072ac98"}

hash_table = pd.DataFrame({
    "file": list(EXPECTED_SHA),
    "sha256_16": [sha16(FEATURE_DIR / f) for f in EXPECTED_SHA],
    "expected (protocol)": list(EXPECTED_SHA.values()),
    "frozen plan": [PLAN_SHA[f] for f in EXPECTED_SHA],
})
hash_table["matches protocol"] = hash_table["sha256_16"] == hash_table["expected (protocol)"]
display(hash_table)
assert hash_table["matches protocol"].all(), "Feature tables differ from the logged hashes: stop and check."

,file,sha256_16,expected (protocol),frozen plan,matches protocol
0,X_train_v3.csv,1cf6b683d2da1516,1cf6b683d2da1516,90e52789a4c4698a,True
1,X_valid_v3.csv,5f880604ffea81e0,5f880604ffea81e0,1a5ebdca6d5f2c2f,True
2,y_train.csv,5174ffcfe33498d9,5174ffcfe33498d9,5174ffcfe33498d9,True
3,y_valid.csv,1d12f28fd072ac98,1d12f28fd072ac98,1d12f28fd072ac98,True


In [3]:
# The 39 features of the final LightGBM, in the saved order
FEATURES = json.loads((MODEL_DIR / "selected_features_v3.json").read_text())["features"]
assert len(FEATURES) == 39 and len(set(FEATURES)) == 39

# Training clients: features (only the 39 columns, in order) and labels
X_train_full = pd.read_csv(FEATURE_DIR / "X_train_v3.csv", index_col="client_id")
y_train_full = pd.read_csv(FEATURE_DIR / "y_train.csv", index_col="client_id")["target"].astype(int)
assert X_train_full.index.equals(y_train_full.index), "X_train and y_train rows are not aligned"
X_train = X_train_full[FEATURES]
y_train = y_train_full
del X_train_full

assert "target" not in X_train.columns and "client_id" not in X_train.columns
assert X_train.shape == (108394, 39)
assert y_train.sum() == 6053

# Test clients: ONLY the ids of X_valid_v3 and the labels file (no features yet)
test_ids = pd.read_csv(FEATURE_DIR / "X_valid_v3.csv", usecols=["client_id"])["client_id"]
y_test_ids = pd.read_csv(FEATURE_DIR / "y_valid.csv", usecols=["client_id"])["client_id"]
assert test_ids.equals(y_test_ids), "Test ids of the NN differ from the LightGBM test ids (y_valid.csv)"
assert len(test_ids) == 27099
assert not set(test_ids) & set(X_train.index), "A client is both in train and test"

print(f"Training clients: {len(X_train):,}, fraud: {y_train.sum():,} ({y_train.mean():.3%})")
print(f"Test clients:     {len(test_ids):,} (ids identical to y_valid.csv, no overlap with train)")

Training clients: 108,394, fraud: 6,053 (5.584%)
Test clients:     27,099 (ids identical to y_valid.csv, no overlap with train)


In [4]:
# A first look at the 39 features of the training clients: type, missing values, range.
# This shows which columns need imputation, which are categorical codes and which are heavy-tailed.
overview = pd.DataFrame({
    "dtype": X_train.dtypes.astype(str),
    "n_unique": X_train.nunique(),
    "missing": X_train.isna().sum(),
    "min": X_train.min(),
    "median": X_train.median(),
    "max": X_train.max(),
    "skew": X_train.skew(),
})
display(overview)

,dtype,n_unique,missing,min,median,max,skew
n_bills,int64,244,0,1.000000,30.000000,3.800000e+02,1.164080
cons_mean,float64,82488,0,0.000000,397.028992,1.174830e+05,24.621526
cons_median,float64,6346,0,0.000000,304.000000,1.174830e+05,28.624985
cons_std,float64,100139,0,0.000000,316.996720,1.637883e+05,38.457766
cons_min,int64,2354,0,0.000000,0.000000,1.174830e+05,52.318327
cons_max,int64,10894,0,0.000000,1317.000000,8.210860e+05,54.618610
consommation_level_1_sum,int64,35387,0,0.000000,10386.500000,1.237815e+06,13.291336
consommation_level_2_sum,int64,14532,0,0.000000,174.000000,2.245578e+06,33.467925
consommation_level_3_sum,int64,6966,0,0.000000,0.000000,4.636550e+05,76.821651
consommation_level_4_sum,int64,7551,0,0.000000,0.000000,2.850472e+06,52.957542


**What the overview shows** (training clients only):

* `tarif_type`, `client_catg` and `region` are **integer codes of categories**, not quantities. A network would read "region 311 > region 101" as a size, so they are one-hot encoded in section 3. Very rare levels are grouped, so the network does not learn from a handful of clients.
* A few columns have **missing values**, for example the gas columns (`cons_std_gaz`, `max_change_gaz`) for clients without a gas meter, and the bill gaps for clients with a single bill. LightGBM handles missing values itself; the network needs them filled, so we use the **median of the training clients**.
* Many consumption columns are **very skewed** (skew far above 1, maximum many times the median). Neural networks train badly on such scales, so we apply `log1p` to skewed non-negative columns and then standardise every column.

## 3 · Preprocessing (fitted on training clients only)

A tree model like LightGBM does not care about the scale of a feature; a neural network does. We therefore build one scikit-learn pipeline:

| Columns | Steps | Learned from the data |
|---|---|---|
| 36 numeric features | median imputation → `log1p` for skewed non-negative columns (skew > 1) → standardisation (mean 0, std 1) | medians, which columns are skewed, means and standard deviations |
| `tarif_type`, `client_catg`, `region` | one-hot encoding; levels with less than 1% of the clients are grouped into one "infrequent" column; unseen levels map to all zeros | the list of levels and which are rare |

**Leakage rule:** everything in the right column is learned only from the clients the network is trained on:

* in the **search** (section 5), the pipeline is fitted on the inner **fit** part (80% of the training clients) and only *applied* to the inner validation part;
* for the **final model** (section 7), it is refitted on all 108,394 training clients and only *applied* to the test clients.

The shared code lives in [nn_pipeline.py](nn_pipeline.py), because the parallel search in section 5 needs it in an importable module (see there). `make_preprocessor()` returns a **fresh, unfitted** pipeline each time, so no fitted state is shared between these uses. We print its source here so the method is visible in the notebook.

In [5]:
import inspect
import nn_pipeline as nnp

CAT_FEATURES = nnp.CAT_FEATURES                                  # tarif_type, client_catg, region
NUM_FEATURES = [f for f in FEATURES if f not in CAT_FEATURES]    # the other 36 columns
assert len(NUM_FEATURES) == 36

print(inspect.getsource(nnp.SkewedLog1p))
print(inspect.getsource(nnp.make_preprocessor))

class SkewedLog1p(BaseEstimator, TransformerMixin):
    # Apply log1p to the columns that are non-negative and strongly right-skewed.
    # Which columns are transformed is decided in fit(), i.e. on the training clients only.
    def __init__(self, skew_threshold=1.0):
        self.skew_threshold = skew_threshold

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        skew = pd.DataFrame(X).skew().to_numpy()
        self.log_mask_ = (X.min(axis=0) >= 0) & (skew > self.skew_threshold)
        self.n_fit_rows_ = X.shape[0]  # kept to check on how many clients it was fitted
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float).copy()
        # clip at 0 so a (rare) negative value in new data cannot produce NaN
        X[:, self.log_mask_] = np.log1p(np.clip(X[:, self.log_mask_], 0, None))
        return X

    def get_feature_names_out(self, input_features=None):
        # one output column per input column, same names
        return

**The inner split.** The network is tuned and stopped early only inside the 108,394 training clients: one stratified 80/20 split with seed 42 (plan: about 86,715 fit / 21,679 validation clients). We build it here and fit the preprocessing on the fit part, to check the output.

In [6]:
# Stratified 80/20 inner split of the TRAINING clients (seed 42). The test clients are not involved.
X_fit, y_fit, X_ival, y_ival = nnp.inner_split(X_train, y_train, seed=SEARCH_SEED)

print(f"Inner fit:        {len(X_fit):,} clients, fraud {y_fit.sum():,} ({y_fit.mean():.4%})")
print(f"Inner validation: {len(X_ival):,} clients, fraud {y_ival.sum():,} ({y_ival.mean():.4%})")
assert len(X_fit) == 86715 and len(X_ival) == 21679

# Fit the preprocessing on the inner FIT clients only, then apply it to both parts
prep_inner = nnp.make_preprocessor(FEATURES).fit(X_fit)
Z_fit = prep_inner.transform(X_fit).astype(np.float32)
Z_ival = prep_inner.transform(X_ival).astype(np.float32)

# Leakage check: every learned step saw exactly the fit clients
assert prep_inner.named_transformers_["num"].named_steps["log"].n_fit_rows_ == len(X_fit)
assert prep_inner.named_transformers_["num"].named_steps["scale"].n_samples_seen_ == len(X_fit)
assert not np.isnan(Z_fit).any() and not np.isnan(Z_ival).any()

log_mask = prep_inner.named_transformers_["num"].named_steps["log"].log_mask_
log_cols = [c for c, m in zip(NUM_FEATURES, log_mask) if m]
print(f"\nNetwork input width: {Z_fit.shape[1]} columns (36 numeric + one-hot columns)")
print(f"log1p applied to {len(log_cols)} of 36 numeric columns:", log_cols)
print("One-hot columns:", [c for c in prep_inner.get_feature_names_out() if c not in NUM_FEATURES])

Inner fit:        86,715 clients, fraud 4,842 (5.5838%)
Inner validation: 21,679 clients, fraud 1,211 (5.5861%)



Network input width: 62 columns (36 numeric + one-hot columns)
log1p applied to 30 of 36 numeric columns: ['n_bills', 'cons_mean', 'cons_median', 'cons_std', 'cons_min', 'cons_max', 'consommation_level_1_sum', 'consommation_level_2_sum', 'consommation_level_3_sum', 'consommation_level_4_sum', 'max_change', 'cons_std_elec', 'max_change_elec', 'cons_std_gaz', 'max_change_gaz', 'bills_per_year', 'gap_mean_days', 'gap_max_days', 'months_number_mean', 'months_number_max', 'n_counters', 'statue_unusual_share', 'remarque_8_share', 'cons_cv', 'level2_3_share', 'level4_share', 'cons_p90_over_median', 'share_last_2y', 'bills_in_2016_17', 'days_since_last_bill']
One-hot columns: ['tarif_type_10', 'tarif_type_11', 'tarif_type_40', 'tarif_type_infrequent_sklearn', 'client_catg_11', 'client_catg_12', 'client_catg_51', 'region_101', 'region_103', 'region_104', 'region_105', 'region_107', 'region_301', 'region_302', 'region_303', 'region_304', 'region_305', 'region_306', 'region_307', 'region_309', '

## 4 · Model design

One **feed-forward MLP** (multilayer perceptron) for binary classification:

* input: the preprocessed features (36 numeric + one-hot columns);
* 1 to 3 hidden layers with the **same number of units**, ReLU activation, **L2** weight penalty and **dropout** after each layer;
* output: one sigmoid unit = predicted fraud probability;
* loss: binary cross-entropy, optimiser Adam;
* **class weights "balanced"** (as for LightGBM): fraud is only 5.6% of the clients, so each fraud case counts about 17 times as much as a non-fraud case in the loss (weights about 8.95 vs 0.53);
* **early stopping** on the inner-validation ROC-AUC (Keras `AUC` metric), patience 10, maximum 100 epochs; the weights of the best epoch are restored.

Search space (frozen plan): 1–3 layers, 32–256 units, dropout 0–0.5, L2 1e-6–1e-3 (log scale), learning rate 1e-4–1e-2 (log scale), batch size 256 / 512 / 1024. Units are drawn on a log scale too, so that small and large networks are tried equally often.

A trial is scored with scikit-learn's **exact** ROC-AUC on the inner validation clients. Keras' `AUC` is an approximation (200 thresholds) and only steers early stopping.

In [7]:
print(inspect.getsource(nnp.build_mlp))
print(inspect.getsource(nnp.fit_with_early_stopping))
print("Class weights on the inner fit clients:", nnp.balanced_class_weight(y_fit))

def build_mlp(n_inputs, n_layers, units, dropout, l2, learning_rate):
    # Feed-forward MLP: n_layers x [Dense(units, ReLU, L2) -> Dropout], sigmoid output
    import keras
    model = keras.Sequential(name="fraud_mlp")
    model.add(keras.Input(shape=(n_inputs,)))
    for _ in range(n_layers):
        model.add(keras.layers.Dense(units, activation="relu",
                                     kernel_regularizer=keras.regularizers.L2(l2)))
        if dropout > 0:
            model.add(keras.layers.Dropout(dropout))
    model.add(keras.layers.Dense(1, activation="sigmoid"))
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=[keras.metrics.AUC(name="auc")],  # Keras' approximate ROC-AUC, steers early stopping
    )
    return model

def fit_with_early_stopping(config, Z_tr, y_tr, Z_va, y_va, seed, max_epochs=100, patience=10, verbose=0):
    # Train one network with early stopping on the validation 

**Smoke test.** Before the search we train one mid-sized network for 3 epochs on the inner split. This checks that the pipeline runs end to end and measures the time per epoch. Its score is **not** used for anything.

In [8]:
smoke_config = {"n_layers": 2, "units": 128, "dropout": 0.2, "l2": 1e-5, "learning_rate": 1e-3, "batch_size": 512}
t0 = time.perf_counter()
smoke_model, _, _, smoke_hist = nnp.fit_with_early_stopping(smoke_config, Z_fit, y_fit, Z_ival, y_ival,
                                                            seed=SEARCH_SEED, max_epochs=3)
sec_per_epoch = (time.perf_counter() - t0) / 3
smoke_model.summary()
print(f"\n{sec_per_epoch:.2f} s per epoch at batch size 512, all cores (includes the one-off graph build)")
print("Inner-validation AUC per epoch (Keras approximation):", [round(v, 4) for v in smoke_hist.history["val_auc"]])
del smoke_model

Model: "fraud_mlp"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         8,064 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 74,117 (289.52 KB)

 Trainable params: 24,705 (96.50 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 49,412 (193.02 KB)


0.93 s per epoch at batch size 512, all cores (includes the one-off graph build)
Inner-validation AUC per epoch (Keras approximation): [0.8374, 0.8481, 0.8554]


## 5 · Random search (30 trials)

**What a trial is:** one randomly drawn combination of settings (layers, units, dropout, L2, learning rate, batch size). The network is trained on the 86,715 inner-fit clients with early stopping on the 21,679 inner-validation clients, and the best inner-validation ROC-AUC is recorded. 30 trials = 30 designs; the best one becomes the final model. 30 matches the 30 settings of the LightGBM search (notebook 09).

**Seeds.** The 30 settings are drawn once with seed 42; every trial trains with seed 42.

**Parallel run.** The networks are small, so one training run uses only part of the 8 CPU cores. We run **4 trials at the same time** in separate processes with 2 CPU threads each (`concurrent.futures`, `spawn`). This does not change any rule of the plan (same 30 settings, same seed, same split). Each worker builds the same inner split and fits the same preprocessing on the inner-fit clients. Because the computations are split differently across threads than in a single process, the scores can differ from a single-process run in the last decimals; with the same setup they are reproducible.

**Resume.** Each finished trial is appended to `results/nn_search_trials.csv` right away. If the run is interrupted, re-running this cell only trains the missing trials.

**Time.** We record the wall-clock time of the whole search (4 workers) and the sum of the single trial times (`fit_seconds`), which is closer to what one process would need.

In [9]:
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing as mp

N_TRIALS, N_WORKERS, THREADS_PER_WORKER = 30, 4, 2
SEARCH_FILE = RESULTS_DIR / "nn_search_trials.csv"
SEARCH_META = RESULTS_DIR / "nn_search_meta.json"   # search timing, used in sections 6 and 8

# The 30 settings, drawn once from the frozen search space with seed 42
configs = nnp.sample_configs(N_TRIALS, seed=SEARCH_SEED)
display(pd.DataFrame(configs))

# Resume: skip the trials that are already in the results file
done = pd.read_csv(SEARCH_FILE) if SEARCH_FILE.exists() else pd.DataFrame()
done_ids = set(done["trial"]) if len(done) else set()
todo = [c for c in configs if c["trial"] not in done_ids]
print(f"{len(done_ids)} trials already done, {len(todo)} to run with {N_WORKERS} workers x {THREADS_PER_WORKER} threads")

,trial,n_layers,units,dropout,l2,learning_rate,batch_size
0,0,1,80,0.429299,0.000124,0.000154,1024
1,1,2,156,0.393032,0.000002,0.000796,1024
2,2,2,220,0.321933,0.000294,0.000771,512
3,3,2,101,0.031909,0.000304,0.001834,256
4,4,1,67,0.485349,0.000478,0.003604,1024
5,5,3,84,0.021902,0.000003,0.002323,256
6,6,3,239,0.162913,0.000013,0.000869,1024
7,7,3,42,0.237852,0.000005,0.002186,256
8,8,3,181,0.350133,0.000009,0.004619,512
9,9,2,72,0.144164,0.000112,0.000190,1024


30 trials already done, 0 to run with 4 workers x 2 threads


In [10]:
search_start = time.perf_counter()
if todo:
    with ProcessPoolExecutor(max_workers=N_WORKERS, mp_context=mp.get_context("spawn"),
                             initializer=nnp.init_search_worker,
                             initargs=(THREADS_PER_WORKER, SEARCH_SEED)) as pool:
        futures = [pool.submit(nnp.run_trial, c) for c in todo]
        for fut in as_completed(futures):
            row = fut.result()
            # append the finished trial right away (resume-safe)
            pd.DataFrame([row]).to_csv(SEARCH_FILE, mode="a", header=not SEARCH_FILE.exists(), index=False)
            print(f"trial {row['trial']:2d}: inner ROC-AUC {row['inner_val_roc_auc']:.6f}, "
                  f"best epoch {row['best_epoch']:3d}/{row['epochs_run']:3d}, {row['fit_seconds']:6.1f} s", flush=True)
search_wall_seconds = time.perf_counter() - search_start

# Save the search timing (only if this run trained trials, so a re-run does not overwrite it with ~0 s)
if todo:
    trials_so_far = pd.read_csv(SEARCH_FILE)
    SEARCH_META.write_text(json.dumps({
        "n_trials": int(len(trials_so_far)),
        "n_workers": N_WORKERS,
        "threads_per_worker": THREADS_PER_WORKER,
        "search_wall_seconds": search_wall_seconds,
        "trials_run_in_this_session": len(todo),
        "sum_trial_fit_seconds": float(trials_so_far["fit_seconds"].sum()),
        "note": "wall time covers only the trials run in this session (resume would split it)",
    }, indent=2))
if todo:
    print(f"Search wall time in this run: {search_wall_seconds:.1f} s")
else:
    print("All 30 trials loaded from results/nn_search_trials.csv (no re-training); original timing:",
          json.loads(SEARCH_META.read_text()))

All 30 trials loaded from results/nn_search_trials.csv (no re-training); original timing: {'n_trials': 30, 'n_workers': 4, 'threads_per_worker': 2, 'search_wall_seconds': 440.88554983300855, 'trials_run_in_this_session': 30, 'sum_trial_fit_seconds': 1582.8153908781242, 'note': 'wall time covers only the trials run in this session (resume would split it)'}


In [11]:
# All 30 trials, sorted from best to worst inner-validation ROC-AUC
search = pd.read_csv(SEARCH_FILE).sort_values("trial").reset_index(drop=True)
assert len(search) == N_TRIALS and search["trial"].is_unique, "search is incomplete"
search.to_csv(SEARCH_FILE, index=False)   # rewrite in trial order (rows were appended in finishing order)

pd.set_option("display.precision", 6)
display(search.sort_values("inner_val_roc_auc", ascending=False).reset_index(drop=True))
print(search[["inner_val_roc_auc", "inner_val_pr_auc", "best_epoch", "fit_seconds"]].describe())
print("\nTrials that used all 100 epochs (did not stop early):", int((~search["stopped_early"]).sum()))

,trial,n_layers,units,dropout,l2,learning_rate,batch_size,best_epoch,epochs_run,stopped_early,inner_val_roc_auc,inner_val_pr_auc,inner_val_keras_auc_best,n_params,fit_seconds,threads,seed
0,1,2,156,0.393032,0.000002,0.000796,1024,32,42,True,0.870165,0.329970,0.870199,34477,60.538820,2,42
1,12,2,85,0.282618,0.000197,0.001860,1024,30,40,True,0.870141,0.329651,0.870165,12751,28.546226,2,42
2,23,2,155,0.359731,0.000020,0.001797,1024,19,29,True,0.870046,0.331966,0.870078,34101,29.367306,2,42
3,2,2,220,0.321933,0.000294,0.000771,512,31,41,True,0.869750,0.327530,0.869706,62701,90.803596,2,42
4,13,2,102,0.151975,0.000001,0.000747,512,25,35,True,0.869610,0.326978,0.869547,17035,36.230959,2,42
5,26,3,219,0.290531,0.000011,0.001520,256,11,21,True,0.868422,0.334018,0.868423,110377,63.911541,2,42
6,6,3,239,0.162913,0.000013,0.000869,1024,13,23,True,0.867696,0.333805,0.867660,130017,62.366875,2,42
7,18,3,136,0.223078,0.000014,0.000401,256,23,33,True,0.867670,0.329952,0.867668,45969,76.335988,2,42
8,9,2,72,0.144164,0.000112,0.000190,1024,95,100,False,0.867646,0.310154,0.867647,9865,68.527709,2,42
9,27,1,235,0.241152,0.000223,0.000146,256,80,90,True,0.867519,0.308976,0.867499,15041,76.668752,2,42


       inner_val_roc_auc  inner_val_pr_auc  best_epoch  fit_seconds
count          30.000000         30.000000   30.000000    30.000000
mean            0.865770          0.314371   43.333333    52.760513
std             0.003137          0.014125   35.968696    30.167148
min             0.858956          0.278825    4.000000    12.417638
25%             0.863574          0.306108   16.750000    30.330174
50%             0.866361          0.315041   27.500000    48.486875
75%             0.867664          0.326857   89.750000    66.660397
max             0.870165          0.334018  100.000000   157.557935

Trials that used all 100 epochs (did not stop early): 8


**How to read the search table.** Each row is one trial. `inner_val_roc_auc` is the exact ROC-AUC on the 21,679 inner-validation clients at the best epoch; `best_epoch` is the epoch with the best (Keras) validation AUC and `epochs_run` how long training ran (best epoch + 10 patience epochs, unless 100 was reached). Trials that never stopped early would suggest that more epochs could help; the summary above counts them.

## 6 · Final configuration

The final configuration is the trial with the **highest inner-validation ROC-AUC** (the plan's selection rule). The number of training epochs for the final refit is fixed to that trial's **best epoch** from the inner validation, because in section 7 the network is trained on all 108,394 training clients, with no validation data left for early stopping.

The configuration is written to `results/nn_final_config.json`. **This file is committed before section 7 runs**, so the test set cannot influence the choice.

In [12]:
best = search.sort_values(["inner_val_roc_auc", "trial"], ascending=[False, True]).iloc[0]
search_meta = json.loads(SEARCH_META.read_text())

final_config = {
    "selected_trial": int(best["trial"]),
    "selection_rule": "highest inner-validation ROC-AUC of 30 random-search trials (inner stratified 80/20 split of the training clients, seed 42)",
    "model": {
        "n_layers": int(best["n_layers"]),
        "units": int(best["units"]),
        "dropout": float(best["dropout"]),
        "l2": float(best["l2"]),
        "learning_rate": float(best["learning_rate"]),
        "batch_size": int(best["batch_size"]),
        "activation": "relu",
        "output": "sigmoid",
        "loss": "binary_crossentropy",
        "optimizer": "adam",
        "class_weight": "balanced",
    },
    "final_epochs": int(best["best_epoch"]),
    "inner_validation": {
        "roc_auc": float(best["inner_val_roc_auc"]),
        "pr_auc": float(best["inner_val_pr_auc"]),
        "best_epoch": int(best["best_epoch"]),
        "epochs_run": int(best["epochs_run"]),
    },
    "preprocessing": "nn_pipeline.make_preprocessor: median imputation, log1p on non-negative columns with skew > 1, StandardScaler; one-hot tarif_type/client_catg/region with levels < 1% grouped; fitted on training clients only",
    "features": "models/selected_features_v3.json (39, in order)",
    "final_seeds": FINAL_SEEDS,
    "search": search_meta,
    "test_set_used": False,
}
(RESULTS_DIR / "nn_final_config.json").write_text(json.dumps(final_config, indent=2))
print(json.dumps(final_config, indent=2))

{
  "selected_trial": 1,
  "selection_rule": "highest inner-validation ROC-AUC of 30 random-search trials (inner stratified 80/20 split of the training clients, seed 42)",
  "model": {
    "n_layers": 2,
    "units": 156,
    "dropout": 0.3930321526384769,
    "l2": 2.4229301715750363e-06,
    "learning_rate": 0.0007957412573105,
    "batch_size": 1024,
    "activation": "relu",
    "output": "sigmoid",
    "loss": "binary_crossentropy",
    "optimizer": "adam",
    "class_weight": "balanced"
  },
  "final_epochs": 32,
  "inner_validation": {
    "roc_auc": 0.8701646944568927,
    "pr_auc": 0.3299697449194004,
    "best_epoch": 32,
    "epochs_run": 42
  },
  "preprocessing": "nn_pipeline.make_preprocessor: median imputation, log1p on non-negative columns with skew > 1, StandardScaler; one-hot tarif_type/client_catg/region with levels < 1% grouped; fitted on training clients only",
  "features": "models/selected_features_v3.json (39, in order)",
  "final_seeds": [
    42,
    1,
    2


---
**Checkpoint.** `results/nn_final_config.json` and `results/nn_search_trials.csv` were committed (`7d84039`) **before** section 7 was added and run. Re-running section 6 rewrites the same file from the same search table; `git diff` on it stays empty.

## 7 · Test evaluation (scored once)

The final configuration from section 6 was committed (`7d84039`) **before** this section ran. Now, and only now, the 27,099 test clients are used:

1. Refit the preprocessing on **all 108,394 training clients** (fresh pipeline, never sees a test client).
2. Train the chosen network on all training clients for exactly `final_epochs` epochs (no validation data, no early stopping), once per seed 42, 1, 2.
3. Load the test features (`X_valid_v3.csv`, 39 features in order), apply the fitted preprocessing and predict.
4. Score each seed **once** and report every number unrounded.

**Metrics** (frozen plan): ROC-AUC (primary), PR-AUC, and the **top-10% inspection list**: the 2,710 clients with the highest scores are flagged; we count fraud caught (of 1,513), hit rate, false alarms and missed fraud. This is the confusion matrix at the plan's operating point; no probability threshold is tuned. Precision, recall and F1 at the **F1-best threshold** are shown only to compare with `model_results.md` (that threshold is picked on the test labels, so it is never used to choose a model).

**LightGBM is not re-scored.** Its numbers are the documented ones from `model_results.md` / notebook 13: ROC-AUC 0.881 and PR-AUC 0.371 (mean of seeds 42, 1, 2), top-10% inspection 864 of 1,513 caught (seed 42).

In [13]:
from sklearn.metrics import precision_recall_curve

final_config = json.loads((RESULTS_DIR / "nn_final_config.json").read_text())
cfg, FINAL_EPOCHS = final_config["model"], final_config["final_epochs"]
assert final_config["test_set_used"] is False
print("Final configuration:", cfg, "| epochs:", FINAL_EPOCHS)

# 1) preprocessing refitted on ALL training clients
prep_final = nnp.make_preprocessor(FEATURES).fit(X_train)
assert prep_final.named_transformers_["num"].named_steps["scale"].n_samples_seen_ == len(X_train) == 108394
Z_train = prep_final.transform(X_train).astype(np.float32)

# 3) the test clients: loaded here for the first time
X_test = pd.read_csv(FEATURE_DIR / "X_valid_v3.csv", index_col="client_id")[FEATURES]
y_test = pd.read_csv(FEATURE_DIR / "y_valid.csv", index_col="client_id")["target"].astype(int)
assert X_test.index.equals(y_test.index) and len(X_test) == 27099 and y_test.sum() == 1513
assert list(X_test.index) == list(test_ids), "test ids differ from section 2"
Z_test = prep_final.transform(X_test).astype(np.float32)   # transform only, nothing is fitted on test

Final configuration: {'n_layers': 2, 'units': 156, 'dropout': 0.3930321526384769, 'l2': 2.4229301715750363e-06, 'learning_rate': 0.0007957412573105, 'batch_size': 1024, 'activation': 'relu', 'output': 'sigmoid', 'loss': 'binary_crossentropy', 'optimizer': 'adam', 'class_weight': 'balanced'} | epochs: 32


In [14]:
TOP_SHARE = 0.10
N_FLAG = int(round(TOP_SHARE * len(y_test)))   # 2,710 inspections
assert N_FLAG == 2710


def top_k_view(y_true, scores, k):
    # Flag the k highest scores: confusion matrix of the inspection list
    order = np.argsort(-scores, kind="stable")
    flagged = np.zeros(len(scores), dtype=bool)
    flagged[order[:k]] = True
    y_true = np.asarray(y_true)
    caught = int((flagged & (y_true == 1)).sum())
    return {"flagged": k, "caught": caught, "missed": int(y_true.sum()) - caught,
            "false_alarms": k - caught, "hit_rate": caught / k, "catch_rate": caught / int(y_true.sum())}, flagged


def f1_best(y_true, scores):
    # Precision, recall, F1 at the F1-best threshold (comparison with model_results.md only)
    p, r, t = precision_recall_curve(y_true, scores)
    f1 = 2 * p * r / np.clip(p + r, 1e-12, None)
    i = int(np.nanargmax(f1[:-1]))
    return {"f1_threshold": float(t[i]), "precision_f1best": float(p[i]), "recall_f1best": float(r[i]), "f1_best": float(f1[i])}


runs, test_scores, flags = [], {}, {}
for seed in FINAL_SEEDS:
    # 2) train for the fixed number of epochs (timed: used in section 8)
    t0 = time.perf_counter()
    model = nnp.fit_fixed_epochs(cfg, Z_train, y_train, seed=seed, epochs=FINAL_EPOCHS)
    fit_seconds = time.perf_counter() - t0
    model.save(MODEL_DIR / f"nn_seed{seed}.keras")   # ignored by git

    # 4) score the test clients once
    s = nnp.predict_proba(model, Z_test)
    test_scores[seed] = s
    top, flags[seed] = top_k_view(y_test, s, N_FLAG)
    runs.append({"seed": seed, "roc_auc": roc_auc_score(y_test, s), "pr_auc": average_precision_score(y_test, s),
                 **{f"top10_{k}": v for k, v in top.items()}, **f1_best(y_test, s),
                 "epochs": FINAL_EPOCHS, "fit_seconds": fit_seconds})
    print(f"seed {seed}: ROC-AUC {runs[-1]['roc_auc']!r}, PR-AUC {runs[-1]['pr_auc']!r}, "
          f"top-10% caught {top['caught']} of 1513, fit {fit_seconds:.1f} s")

runs = pd.DataFrame(runs)

seed 42: ROC-AUC 0.8735311838425354, PR-AUC 0.3493123660607573, top-10% caught 852 of 1513, fit 11.3 s


seed 1: ROC-AUC 0.8730556289328955, PR-AUC 0.3505757861600036, top-10% caught 851 of 1513, fit 15.4 s


seed 2: ROC-AUC 0.8742450651378095, PR-AUC 0.35087473512745504, top-10% caught 860 of 1513, fit 15.7 s


In [15]:
# Summary over the 3 seeds (std = sample standard deviation, ddof=1)
metric_cols = [c for c in runs.columns if c not in ("seed", "epochs")]
summary = pd.DataFrame({stat: getattr(runs[metric_cols], stat)() for stat in ["mean", "min", "max", "std"]}).T
summary.insert(0, "seed", summary.index)
final_runs = pd.concat([runs.assign(seed=runs["seed"].astype(str)), summary.reset_index(drop=True)], ignore_index=True)
final_runs.to_csv(RESULTS_DIR / "nn_final_runs.csv", index=False)

pd.set_option("display.precision", 10)
display(final_runs.set_index("seed").T)

seed,42,1,2,mean,min,max,std
roc_auc,0.8735311838,0.8730556289,0.8742450651,0.8736106260,0.8730556289,0.8742450651,0.0005986843
pr_auc,0.3493123661,0.3505757862,0.3508747351,0.3502542958,0.3493123661,0.3508747351,0.0008293168
top10_flagged,2710.0000000000,2710.0000000000,2710.0000000000,2710.0000000000,2710.0000000000,2710.0000000000,0.0000000000
top10_caught,852.0000000000,851.0000000000,860.0000000000,854.3333333333,851.0000000000,860.0000000000,4.9328828623
top10_missed,661.0000000000,662.0000000000,653.0000000000,658.6666666667,653.0000000000,662.0000000000,4.9328828623
top10_false_alarms,1858.0000000000,1859.0000000000,1850.0000000000,1855.6666666667,1850.0000000000,1859.0000000000,4.9328828623
top10_hit_rate,0.3143911439,0.3140221402,0.3173431734,0.3152521525,0.3140221402,0.3173431734,0.0018202520
top10_catch_rate,0.5631196299,0.5624586913,0.5684071381,0.5646618198,0.5624586913,0.5684071381,0.0032603324
f1_threshold,0.7947425246,0.8108853102,0.8031262159,0.8029180169,0.7947425246,0.8108853102,0.0080734064
precision_f1best,0.3497942387,0.3706583722,0.3630069238,0.3611531782,0.3497942387,0.3706583722,0.0105548707


### Comparison with the documented LightGBM

The decision rule of the frozen plan: the NN is favoured only if its **mean** test ROC-AUC is **at least 0.005 above 0.881**, the gap is **larger than its seed spread**, and PR-AUC and the top-10% catch are **not worse**. LightGBM is favoured if the NN is below 0.886, within the seed spread, or worse on PR-AUC or the top-10% catch.

The LightGBM values are the documented, rounded ones (3 decimals) from `model_results.md`. The re-run of 2026-11-05 (protocol.md, *not* the reported baseline) gave 0.8805 / 0.3707, so the rounding is about ±0.0005.

In [16]:
LGBM = {"roc_auc": 0.881, "pr_auc": 0.371, "top10_caught": 864}   # documented, not re-scored

nn_mean = runs[["roc_auc", "pr_auc", "top10_caught"]].mean()
seed_spread = runs["roc_auc"].max() - runs["roc_auc"].min()
comparison = pd.DataFrame({
    "LightGBM (documented)": LGBM,
    "NN mean of 3 seeds": nn_mean,
    "NN min": runs[["roc_auc", "pr_auc", "top10_caught"]].min(),
    "NN max": runs[["roc_auc", "pr_auc", "top10_caught"]].max(),
})
comparison["NN - LightGBM (absolute)"] = comparison["NN mean of 3 seeds"] - comparison["LightGBM (documented)"]
comparison["NN vs LightGBM (relative)"] = comparison["NN - LightGBM (absolute)"] / comparison["LightGBM (documented)"]
display(comparison)

roc_gap = nn_mean["roc_auc"] - LGBM["roc_auc"]
rule = {
    "roc_gap_at_least_0.005": bool(roc_gap >= 0.005),
    "roc_gap_larger_than_seed_spread": bool(abs(roc_gap) > seed_spread),
    "pr_auc_not_worse": bool(nn_mean["pr_auc"] >= LGBM["pr_auc"]),
    "top10_not_worse": bool(nn_mean["top10_caught"] >= LGBM["top10_caught"]),
}
print(f"ROC-AUC gap (NN mean - LightGBM): {roc_gap!r}; NN seed spread (max - min): {seed_spread!r}")
for k, v in rule.items():
    print(f"  {k}: {v}")
favours_nn = rule["roc_gap_at_least_0.005"] and rule["roc_gap_larger_than_seed_spread"] and rule["pr_auc_not_worse"] and rule["top10_not_worse"]
print("\nDecision rule favours the neural network:", favours_nn)

,LightGBM (documented),NN mean of 3 seeds,NN min,NN max,NN - LightGBM (absolute),NN vs LightGBM (relative)
roc_auc,0.881,0.8736106260,0.8730556289,0.8742450651,-0.0073893740,-0.0083874847
pr_auc,0.371,0.3502542958,0.3493123661,0.3508747351,-0.0207457042,-0.0559183402
top10_caught,864.000,854.3333333333,851.0000000000,860.0000000000,-9.6666666667,-0.0111882716


ROC-AUC gap (NN mean - LightGBM): -0.007389374028919926; NN seed spread (max - min): 0.0011894362049140739
  roc_gap_at_least_0.005: False
  roc_gap_larger_than_seed_spread: True
  pr_auc_not_worse: False
  top10_not_worse: False

Decision rule favours the neural network: False


### Costly errors (descriptive only)

For seed 42 we look at the two costly error types of the top-10% list: **missed fraud** (lost revenue) and **false alarms** (wasted visits). We compare the raw feature medians of caught fraud, missed fraud and false alarms. This only describes the errors as the handout asks; it is **not** a subgroup evaluation (team decision 2026-11-05) and nothing is tuned on it.

In [17]:
seed_view = 42
flag = pd.Series(flags[seed_view], index=X_test.index)
groups = {
    "caught fraud":  flag & (y_test == 1),
    "missed fraud":  ~flag & (y_test == 1),
    "false alarms":  flag & (y_test == 0),
    "correctly not flagged": ~flag & (y_test == 0),
}
print({k: int(v.sum()) for k, v in groups.items()})
error_profile = pd.DataFrame({k: X_test[v].median() for k, v in groups.items()})
display(error_profile.loc[["n_bills", "cons_mean", "cons_median", "history_years", "client_age_years",
                           "days_since_last_bill", "n_counters", "share_last_2y", "cons_cv",
                           "statue_unusual_share", "n_codes", "level4_share"]])

# Score distribution of the missed fraud: are they just below the cut-off or far away?
cut = np.sort(test_scores[seed_view])[::-1][N_FLAG - 1]
missed_scores = test_scores[seed_view][groups["missed fraud"].to_numpy()]
print(f"Top-10% cut-off score (seed 42): {cut!r}")
print("Missed fraud scores, quantiles:", np.quantile(missed_scores, [0.1, 0.25, 0.5, 0.75, 0.9]))

{'caught fraud': 852, 'missed fraud': 661, 'false alarms': 1858, 'correctly not flagged': 23728}


,caught fraud,missed fraud,false alarms,correctly not flagged
n_bills,49.0000000000,38.0000000000,46.0000000000,26.0000000000
cons_mean,449.1798048048,443.8750000000,467.4676470588,385.1062030075
cons_median,293.2500000000,302.0000000000,308.2500000000,301.7500000000
history_years,13.2279260780,10.9897330595,13.1444216290,8.0629705681
client_age_years,20.1779603012,13.2101300479,18.1793292266,13.2826830938
days_since_last_bill,224.0000000000,251.0000000000,223.0000000000,244.0000000000
n_counters,2.5000000000,2.0000000000,2.0000000000,1.0000000000
share_last_2y,0.1607142857,0.1904761905,0.1623310811,0.2500000000
cons_cv,1.0073916252,0.9827111753,1.0329664546,0.8385519093
statue_unusual_share,0.0000000000,0.0000000000,0.0134234234,0.0000000000


Top-10% cut-off score (seed 42): 0.75847983
Missed fraud scores, quantiles: [0.17825654 0.35759383 0.5471639  0.67435259 0.73212749]


## 8 · Timing

Same machine (Apple M2, 8 cores, CPU only) for both models, but two Python environments, so the numbers are indicative.

* **Training:** one final NN fit = the seed-42 fit of section 7 (all 108,394 clients, fixed epochs). Total search effort from `results/nn_search_meta.json` (wall time with 4 workers, and the summed single-trial times).
* **Inference:** the 27,099 test clients in one batch, **1 warm-up + 10 timed repetitions**, median and range per 1,000 clients. For the NN we time the model alone **and** preprocessing + model (what serving needs; LightGBM needs no preprocessing).
* **Size:** the saved model file on disk (the NN also needs the fitted preprocessing; its pickled size is reported).
* **LightGBM:** `lightgbm_timing.py` runs in the main environment (`.venv`): one fit with `models/lightgbm_final_params.json`, inference with `models/lightgbm_final.txt`. **Timing only, no metric.**

In [18]:
import pickle
import resource
import subprocess

N_REPEATS = 10
nn_model = keras.models.load_model(MODEL_DIR / "nn_seed42.keras")


def time_calls(fn, n_repeats=N_REPEATS):
    fn()                                   # warm-up (graph build, caches), not timed
    t = []
    for _ in range(n_repeats):
        t0 = time.perf_counter()
        fn()
        t.append(time.perf_counter() - t0)
    return np.array(t)


n_test = len(X_test)
t_model = time_calls(lambda: nnp.predict_proba(nn_model, Z_test))
t_e2e = time_calls(lambda: nnp.predict_proba(nn_model, prep_final.transform(X_test).astype(np.float32)))
prep_bytes = len(pickle.dumps(prep_final))
nn_file_bytes = (MODEL_DIR / "nn_seed42.keras").stat().st_size

search_meta = json.loads(SEARCH_META.read_text())
nn_timing = {
    "model": "Neural network (MLP, final config, seed 42)",
    "final_fit_seconds": float(runs.loc[runs["seed"] == 42, "fit_seconds"].iloc[0]),
    "final_fit_seconds_all_seeds": runs["fit_seconds"].tolist(),
    "search_wall_seconds": search_meta["search_wall_seconds"],
    "search_sum_trial_fit_seconds": search_meta["sum_trial_fit_seconds"],
    "search_trials": search_meta["n_trials"],
    "search_workers_x_threads": f'{search_meta["n_workers"]}x{search_meta["threads_per_worker"]}',
    "threads": "all (8)",
    "n_test": n_test,
    "inference_repeats": N_REPEATS,
    "inference_batch_seconds_median": float(np.median(t_model)),
    "inference_ms_per_1000_median": float(np.median(t_model) / n_test * 1e6),
    "inference_ms_per_1000_min": float(t_model.min() / n_test * 1e6),
    "inference_ms_per_1000_max": float(t_model.max() / n_test * 1e6),
    "inference_with_preprocessing_ms_per_1000_median": float(np.median(t_e2e) / n_test * 1e6),
    "inference_with_preprocessing_ms_per_1000_min": float(t_e2e.min() / n_test * 1e6),
    "inference_with_preprocessing_ms_per_1000_max": float(t_e2e.max() / n_test * 1e6),
    "throughput_clients_per_second": float(n_test / np.median(t_e2e)),
    "model_file": "models/nn_seed42.keras",
    "model_file_bytes": nn_file_bytes,
    "preprocessor_pickle_bytes": prep_bytes,
    "peak_rss_mb": resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024 / 1024,
    "note": "peak RSS is the whole notebook process (data, search bookkeeping, 3 fits): an upper bound",
}

# LightGBM: separate process in the main environment, timing only
out = subprocess.run([str(Path(".venv/bin/python")), "lightgbm_timing.py"], capture_output=True, text=True, check=True)
lgbm_timing = json.loads(out.stdout.strip().splitlines()[-1])

timing = pd.DataFrame([nn_timing, lgbm_timing]).set_index("model").T
timing.to_csv(RESULTS_DIR / "nn_timing.csv")
display(timing)

model,"Neural network (MLP, final config, seed 42)","LightGBM (final, feature set v3)"
final_fit_seconds,11.277063417,7.173105791
final_fit_seconds_all_seeds,"[11.27706341695739, 15.431997250008862, 15.663...",NaN
search_wall_seconds,440.885549833,NaN
search_sum_trial_fit_seconds,1582.8153908781,NaN
search_trials,30.0,NaN
search_workers_x_threads,4x2,NaN
threads,all (8),8
n_test,27099,27099
inference_repeats,10,10
inference_batch_seconds_median,0.0553975835,0.2891397915


**Reading the timing table.** Both models score all 27,099 clients in well under a second, so for a batch inspection list (the plan's assumption) inference speed does not decide between them. The larger differences are training effort (a 30-trial search with early stopping vs. a single boosting fit) and what has to be deployed: the NN needs the fitted preprocessing pipeline and a TensorFlow runtime in a separate Python 3.12 environment, LightGBM one text model file.

## 9 · Record of the environment

Library versions of both environments, hardware, seeds and the total runtime of this notebook, so the results can be traced.

In [19]:
env = {
    "date": time.strftime("%Y-%m-%d"),
    "hardware": {"machine": platform.machine(), "processor": "Apple M2", "cpu_cores": os.cpu_count(),
                 "ram_gb": 8, "os": f"macOS {platform.mac_ver()[0]}", "accelerator": "none (CPU only, no Metal)"},
    "nn_environment": {"path": ".venv-nn", "python": platform.python_version(), "tensorflow": tf.__version__,
                       "keras": keras.__version__, "scikit-learn": sklearn.__version__,
                       "pandas": pd.__version__, "numpy": np.__version__},
    "lightgbm_environment": {"path": ".venv", "python": lgbm_timing["python"], "lightgbm": lgbm_timing["lightgbm_version"]},
    "seeds": {"search": SEARCH_SEED, "final": FINAL_SEEDS},
    "op_determinism": True,
    "search": search_meta,
    "final_config_file": "results/nn_final_config.json",
    "feature_table_sha256_16": dict(zip(hash_table["file"], hash_table["sha256_16"])),
    "notebook_runtime_seconds_this_run": time.perf_counter() - NOTEBOOK_START,
    "total_nn_compute_seconds": search_meta["search_wall_seconds"] + float(runs["fit_seconds"].sum()),
    "note": "notebook runtime of this run excludes the search when it was loaded from results/nn_search_trials.csv",
}
(RESULTS_DIR / "nn_environment.json").write_text(json.dumps(env, indent=2))
print(json.dumps(env, indent=2))

{
  "date": "2026-11-05",
  "hardware": {
    "machine": "arm64",
    "processor": "Apple M2",
    "cpu_cores": 8,
    "ram_gb": 8,
    "os": "macOS 26.2",
    "accelerator": "none (CPU only, no Metal)"
  },
  "nn_environment": {
    "path": ".venv-nn",
    "python": "3.12.14",
    "tensorflow": "2.16.2",
    "keras": "3.15.1",
    "scikit-learn": "1.9.1",
    "pandas": "3.0.6",
    "numpy": "1.26.4"
  },
  "lightgbm_environment": {
    "path": ".venv",
    "python": "3.13.18",
    "lightgbm": "4.7.0"
  },
  "seeds": {
    "search": 42,
    "final": [
      42,
      1,
      2
    ]
  },
  "op_determinism": true,
  "search": {
    "n_trials": 30,
    "n_workers": 4,
    "threads_per_worker": 2,
    "search_wall_seconds": 440.88554983300855,
    "trials_run_in_this_session": 30,
    "sum_trial_fit_seconds": 1582.8153908781242,
    "note": "wall time covers only the trials run in this session (resume would split it)"
  },
  "final_config_file": "results/nn_final_config.json",
  "feature